<a href="https://colab.research.google.com/github/everestso/everestso/blob/main/LLM_2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP + FastMCP (Colab-Friendly) Demo: Summarize a GitHub README

**Goal:** Demonstrate MCP tool registration + tool discovery + tool calling inside a *single* Colab notebook.

We use **FastMCP** to create an MCP server **in-memory** (same Python process).
This avoids Colab/Jupyter issues with subprocess pipes (`fileno`) and event loops.

Later, we’ll connect an LLM to these tools for “tool-using” summarization.

## Several URL's
target_url = "https://github.com/numpy/numpy"



In [ ]:
target_url = "https://github.com/numpy/numpy"
target_url = "https://github.com/psf/requests"

In [ ]:
# Cell 2 — Install dependencies

!pip -q install fastmcp httpx


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.3/413.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.4/197.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

In [ ]:
# Cell 3 — Imports + quick config

import httpx
from fastmcp import FastMCP, Client

target_url = "https://github.com/psf/requests"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 4 — Create an MCP server (in-memory) and define a tool

# Create a small MCP server object
mcp = FastMCP("GitHub README Tools")

@mcp.tool()
def get_github_readme(repo_url: str) -> str:
    """
    Fetch README.md from a public GitHub repo.
    Teaching note: GitHub repos may use 'main' or 'master' as default branch.
    We'll try main first, then master as a fallback.
    """
    base = repo_url.replace("https://github.com/", "https://raw.githubusercontent.com/")

    # Try common default branches
    candidates = [
        f"{base}/main/README.md",
        f"{base}/master/README.md",
        f"{base}/README.md",
    ]

    for raw_url in candidates:
        r = httpx.get(raw_url, timeout=20)
        if r.status_code == 200:
            return r.text

    return (
        "README not found at common locations.\n"
        "Tried:\n- " + "\n- ".join(candidates)
    )


In [ ]:
# Cell 5 — Connect an MCP client (in-memory) and discover tools

mcp_client = Client(mcp)

async with mcp_client:
    tools = await mcp_client.list_tools()

# Print tool names (what the server exposes)
print("Discovered MCP tools:")
for t in tools:
    print("-", t.name)


Discovered MCP tools:
- get_github_readme


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 6 — Call the MCP tool directly (no LLM yet)
## START HERE for RERUN

async with mcp_client:
    result = await mcp_client.call_tool(
        "get_github_readme",
        {"repo_url": target_url}
    )

# MCP tool results are structured content; first chunk is usually text
readme_text = result.content[0].text

print("README preview (first 800 chars):\n")
print(readme_text[:800])


README preview (first 800 chars):

# Requests

**Requests** is a simple, yet elegant, HTTP library.

```python
>>> import requests
>>> r = requests.get('https://httpbin.org/basic-auth/user/pass', auth=('user', 'pass'))
>>> r.status_code
200
>>> r.headers['content-type']
'application/json; charset=utf8'
>>> r.encoding
'utf-8'
>>> r.text
'{"authenticated": true, ...'
>>> r.json()
{'authenticated': True, ...}
```

Requests allows you to send HTTP/1.1 requests extremely easily. There’s no need to manually add query strings to your URLs, or to form-encode your `PUT` & `POST` data — but nowadays, just use the `json` method!

Requests is one of the most downloaded Python packages today, pulling in around `30M downloads / week`— according to GitHub, Requests is currently [depended upon](https://github.com/psf/requests/network/depen


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 7 — Build a summarization prompt (string construction demo)

prompt = (
    "Summarize the README for the following GitHub repository:\n"
    + target_url
)

print(prompt)


Summarize the README for the following GitHub repository:
https://github.com/psf/requests


In [ ]:
# Cell 8 — Simple “local summary” placeholder (to prove the pipeline)

# Placeholder summary logic (non-LLM): grab first few lines and headings
lines = readme_text.splitlines()
headings = [ln for ln in lines if ln.strip().startswith("#")][:10]
preview = "\n".join(lines[:30])

print("=== Placeholder Summary (non-LLM) ===\n")
print("Headings found:")
print("\n".join(headings) if headings else "(none detected)")
print("\nFirst ~30 lines preview:\n")
print(preview)


=== Placeholder Summary (non-LLM) ===

Headings found:
# Requests
## Installing Requests and Supported Versions
## Supported Features & Best–Practices
## API Reference and User Guide available on [Read the Docs](https://requests.readthedocs.io)
## Cloning the repository

First ~30 lines preview:

# Requests

**Requests** is a simple, yet elegant, HTTP library.

```python
>>> import requests
>>> r = requests.get('https://httpbin.org/basic-auth/user/pass', auth=('user', 'pass'))
>>> r.status_code
200
>>> r.headers['content-type']
'application/json; charset=utf8'
>>> r.encoding
'utf-8'
>>> r.text
'{"authenticated": true, ...'
>>> r.json()
{'authenticated': True, ...}
```

Requests allows you to send HTTP/1.1 requests extremely easily. There’s no need to manually add query strings to your URLs, or to form-encode your `PUT` & `POST` data — but nowadays, just use the `json` method!

Requests is one of the most downloaded Python packages today, pulling in around `30M downloads / week`— accord

#  Now the LLM

## Important Note: Different LLMs Behave Differently with Tools

In this notebook, we intentionally separate **MCP tool usage** from **LLM behavior**.

Even when using the *same code* and *same tools*, different large language models
behave differently:

### OpenAI (e.g., `gpt-4o-mini`)
- Generally **eager** to use tools
- More likely to automatically call a tool when `tool_choice="auto"`
- Often a good first choice for demonstrating agentic behavior

### Anthropic Claude (e.g., `claude-sonnet`)
- More **conservative and cautious**
- May describe what it *would* do instead of calling a tool
- Often benefits from **explicit instructions** like:
  > “Use the available tool to retrieve the README, then summarize it.”

### Open-source / routed models (e.g., LLaMA via Groq)
- Tool support depends on the **provider**, not just the model
- Some models accept tool schemas but do not reliably invoke them
- Behavior can vary across providers even with the same model name

### Key takeaway
**MCP standardizes tools — not model behavior.**

Tool discovery and execution are consistent, but
**LLMs still make decisions**, and those decisions vary by model.

This is why robust agent systems:
- check whether tools were actually called
- use fallback logic
- and sometimes separate “planning” from “execution”


In [ ]:
# Cell 1 — Install the Anthropic SDK

!pip -q install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 4.2 MB/s eta 0:00:00


In [ ]:
# Cell 2 — Configure the Anthropic client

import os
from google.colab import userdata
import anthropic

# Make sure your key is set:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 3 — Summarize the README using Claude (native API)

model_name = "claude-sonnet-4-5-20250929"

prompt = f"""
You are helping students quickly understand a GitHub repository.

Summarize the README content below in:
1) What this library is (1–2 sentences)
2) Typical use cases (bullets)
3) How to install (bullets)
4) Any cautions or best practices (bullets)

Keep the explanation concise and student-friendly.

README:
{readme_text}
"""

response = client.messages.create(
    model=model_name,
    max_tokens=600,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)


# Requests Library - Quick Summary

## 1) What this library is
Requests is a simple and elegant Python library for making HTTP requests. It simplifies sending HTTP/1.1 requests without needing to manually handle query strings, form encoding, or complex configurations.

## 2) Typical use cases
- Making GET/POST/PUT requests to web APIs
- Sending JSON data to REST APIs
- Downloading files from the web
- Authenticating with web services (Basic/Digest auth)
- Managing sessions with cookies
- Uploading files to servers
- Working with proxies and SSL/TLS verification

## 3) How to install
- Install via pip: `python -m pip install requests`
- Requires Python 3.9 or higher
- If cloning the repo, use: `git clone -c fetch.fsck.badTimezone=ignore https://github.com/psf/requests.git`

## 4) Cautions or best practices
- **Use the `.json()` method** for sending/receiving JSON data (instead of manual form-encoding)
- **Set connection timeouts** to prevent hanging requests
- **Leverage sessions** for 

# Simple Re-Run

In [ ]:
target_url = "https://github.com/numpy/numpy"

In [ ]:
# Cell 6 — Call the MCP tool directly (no LLM yet)
## START HERE for RERUN

async with mcp_client:
    result = await mcp_client.call_tool(
        "get_github_readme",
        {"repo_url": target_url}
    )

# MCP tool results are structured content; first chunk is usually text
readme_text = result.content[0].text

print("README preview (first 800 chars):\n")
print(readme_text[:800])


README preview (first 800 chars):

<h1 align="center">
<img src="https://raw.githubusercontent.com/numpy/numpy/main/branding/logo/primary/numpylogo.svg" width="300">
</h1><br>


[![Powered by NumFOCUS](https://img.shields.io/badge/powered%20by-NumFOCUS-orange.svg?style=flat&colorA=E1523D&colorB=007D8A)](
https://numfocus.org)
[![PyPI Downloads](https://img.shields.io/pypi/dm/numpy.svg?label=PyPI%20downloads)](
https://pypi.org/project/numpy/)
[![Conda Downloads](https://img.shields.io/conda/dn/conda-forge/numpy.svg?label=Conda%20downloads)](
https://anaconda.org/conda-forge/numpy)
[![Stack Overflow](https://img.shields.io/badge/stackoverflow-Ask%20questions-blue.svg)](
https://stackoverflow.com/questions/tagged/numpy)
[![Nature Paper](https://img.shields.io/badge/DOI-10.1038%2Fs41586--020--2649--2-blue)](
https://doi.org/10


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 3 — Summarize the README using Claude (native API)

model_name = "claude-sonnet-4-5-20250929"

prompt = f"""
You are helping students quickly understand a GitHub repository.

Summarize the README content below in:
1) What this library is (1–2 sentences)
2) Typical use cases (bullets)
3) How to install (bullets)
4) Any cautions or best practices (bullets)

Keep the explanation concise and student-friendly.

README:
{readme_text}
"""

response = client.messages.create(
    model=model_name,
    max_tokens=600,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)


# NumPy Quick Summary

## 1) What this library is
NumPy is the fundamental Python package for scientific computing that provides powerful N-dimensional arrays and mathematical functions. It's the foundation for data manipulation, numerical calculations, and scientific operations in Python.

## 2) Typical use cases
- Working with large multi-dimensional arrays and matrices
- Performing mathematical and logical operations on arrays
- Linear algebra computations
- Fourier transforms
- Random number generation
- Integrating C/C++/Fortran code with Python
- Foundation for data science libraries (pandas, scikit-learn, etc.)

## 3) How to install
- **Via pip:** `pip install numpy`
- **Via conda:** `conda install numpy` (from conda-forge channel)
- Check the official documentation at https://numpy.org for detailed installation instructions

## 4) Cautions or best practices
- **Testing required:** If contributing or running tests, you'll need `pytest` and `hypothesis` installed
- **Community-dr

# Automatic Tool Call

🔁 Part 3 — Automatic Tool Call (Claude decides)

Prerequisite:
You already have:

mcp (FastMCP server)

mcp_client = Client(mcp)

get_github_readme tool defined

Native Anthropic client configured

In [ ]:
target_url = "https://github.com/numpy/numpy"

In [ ]:
# Cell 1 — Describe the MCP tool in Anthropic’s tool format

anthropic_tools = [
    {
        "name": "get_github_readme",
        "description": "Fetch the README.md file from a public GitHub repository.",
        "input_schema": {
            "type": "object",
            "properties": {
                "repo_url": {
                    "type": "string",
                    "description": "GitHub repository URL, e.g. https://github.com/psf/requests"
                }
            },
            "required": ["repo_url"]
        }
    }
]


In [ ]:
# Cell 2 — Ask Claude a question where a tool is useful

model_name = "claude-sonnet-4-5-20250929"

agent_prompt = f"""
You are an assistant that can use tools.

Task:
Summarize the GitHub repository at the following URL for a student.

If you need the README content to answer accurately,
use the available tool to retrieve it.

GitHub repository:
{target_url}
"""


In [ ]:
# Cell 3 — Claude decides whether to call the tool

response = client.messages.create(
    model=model_name,
    max_tokens=600,
    tools=anthropic_tools,
    messages=[
        {"role": "user", "content": agent_prompt}
    ]
)



In [ ]:
# Cell 4 — Check whether Claude requested a tool call
## This is the critical agentic step.

tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

if not tool_use_blocks:
    print("Claude did not request a tool call.")
    print(response.content[0].text)
else:
    tool_use = tool_use_blocks[0]
    print("Tool requested:", tool_use.name)
    print("Tool input:", tool_use.input)
    print("Tool use id:", tool_use.id)



Tool requested: get_github_readme
Tool input: {'repo_url': 'https://github.com/numpy/numpy'}
Tool use id: toolu_01CYdQnpeo9fppyfEYwG136L


In [ ]:
# Cell 5 — Execute the tool (only if Claude asked)

if tool_use:
    async with mcp_client:
        tool_result = await mcp_client.call_tool(
            tool_use.name,
            tool_use.input
        )

    readme_text = tool_result.content[0].text



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 6 — Send tool result back to Claude for final answer

final_response = client.messages.create(
    model=model_name,
    max_tokens=600,
    tools=anthropic_tools,
    messages=[
        {"role": "user", "content": agent_prompt},
        {"role": "assistant", "content": response.content},  # includes tool_use block
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": readme_text
                }
            ]
        }
    ]
)

# Print final text blocks
for block in final_response.content:
    if block.type == "text":
        print(block.text)



## Summary of NumPy for Students

**What is NumPy?**
NumPy (Numerical Python) is the fundamental package for scientific computing in Python. It's one of the most important libraries you'll use in data science, mathematics, engineering, and scientific programming.

**Key Features:**
- **N-dimensional arrays**: Provides powerful array objects that are much faster than Python lists for numerical operations
- **Broadcasting functions**: Allows mathematical operations on arrays of different shapes
- **Integration tools**: Helps connect Python with C/C++ and Fortran code for better performance
- **Scientific functions**: Includes linear algebra, Fourier transforms, and random number generation capabilities

**Why Students Should Learn NumPy:**
- It's the foundation for other popular libraries like Pandas, SciPy, Matplotlib, and scikit-learn
- Essential for data analysis, machine learning, and scientific computing courses
- Used extensively in academic research and industry
- Fast and efficie

# Agent Loop

## Part 3 — Agent Loop (Claude + MCP Tools)

In this section, Claude can:
- decide when it needs a tool
- request one or more tool calls
- receive tool results
- repeat until it can produce a final answer

This is the core pattern behind many “agentic” systems.


In [ ]:
# Cell 1 — Ensure tools are defined for Claude (Anthropic tool schema)

anthropic_tools = [
    {
        "name": "get_github_readme",
        "description": "Fetch README content from a public GitHub repository.",
        "input_schema": {
            "type": "object",
            "properties": {
                "repo_url": {
                    "type": "string",
                    "description": "GitHub repo URL, e.g. https://github.com/psf/requests"
                }
            },
            "required": ["repo_url"],
        },
    }
]


In [ ]:
# Cell 2 — Agent loop function (multi-step, multi-tool)

from typing import Any, Dict, List, Optional

def extract_tool_uses(content_blocks) -> List[Any]:
    """Return all tool_use blocks from Anthropic response content."""
    return [b for b in content_blocks if getattr(b, "type", None) == "tool_use"]

def extract_text(content_blocks) -> str:
    """Concatenate any text blocks from Anthropic response content."""
    parts = []
    for b in content_blocks:
        if getattr(b, "type", None) == "text":
            parts.append(b.text)
    return "\n".join(parts).strip()

async def run_claude_mcp_agent_loop(
    *,
    anthropic_client,
    model: str,
    user_prompt: str,
    tools: List[Dict[str, Any]],
    mcp_client,
    max_steps: int = 6,
    max_tokens: int = 700,
    verbose: bool = True,
) -> str:
    """
    Agent loop:
    1) Call Claude with tools available.
    2) If Claude requests tool_use blocks, execute each via MCP.
    3) Return tool_result blocks (must be the immediate next message).
    4) Repeat until Claude stops requesting tools or max_steps reached.
    """

    # Conversation transcript (Anthropic format)
    messages: List[Dict[str, Any]] = [{"role": "user", "content": user_prompt}]

    async with mcp_client:
        for step in range(1, max_steps + 1):
            if verbose:
                print(f"\n=== Agent step {step} ===")

            # Ask Claude
            resp = anthropic_client.messages.create(
                model=model,
                max_tokens=max_tokens,
                tools=tools,
                messages=messages,
            )

            # Add Claude response to transcript
            messages.append({"role": "assistant", "content": resp.content})

            # Check for tool requests
            tool_uses = extract_tool_uses(resp.content)

            if not tool_uses:
                # Done: return final assistant text
                final_text = extract_text(resp.content)
                if verbose:
                    print("No tool requests. Finishing.")
                return final_text if final_text else "(No text returned.)"

            if verbose:
                print(f"Tool requests: {len(tool_uses)}")

            # Execute all requested tools and construct tool_result blocks
            tool_results_payload = []
            for tu in tool_uses:
                tool_name = tu.name
                tool_input = tu.input
                tool_use_id = tu.id

                if verbose:
                    print(f"- Calling tool: {tool_name}  input={tool_input}")

                try:
                    mcp_result = await mcp_client.call_tool(tool_name, tool_input)
                    result_text = mcp_result.content[0].text
                except Exception as e:
                    result_text = f"Tool error calling {tool_name}: {e}"

                tool_results_payload.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": result_text
                    }
                )

            # IMPORTANT: tool_result blocks must be the next message after tool_use
            messages.append({"role": "user", "content": tool_results_payload})

        return "Stopped: reached max_steps without finishing. Increase max_steps or simplify the task."


In [ ]:
# Cell 3 — Run the agent loop

target_url = "https://github.com/psf/requests"
model_name = "claude-sonnet-4-5-20250929"

user_prompt = f"""
You are an assistant that can use tools.

Goal: Produce a short, student-friendly summary of this GitHub repository.

Repository:
{target_url}

Rules:
- If you need repository content, use the available tool(s).
- Keep the final answer under ~200 words.

Output format:
1) What it is (1–2 sentences)
2) Typical use cases (bullets)
3) How to install (bullets)
4) Best practices / cautions (bullets)
"""

final_answer = await run_claude_mcp_agent_loop(
    anthropic_client=client,
    model=model_name,
    user_prompt=user_prompt,
    tools=anthropic_tools,
    mcp_client=mcp_client,
    max_steps=6,
    max_tokens=700,
    verbose=True,
)

print("\n=== Final Answer ===\n")
print(final_answer)



=== Agent step 1 ===
Tool requests: 1
- Calling tool: get_github_readme  input={'repo_url': 'https://github.com/psf/requests'}

=== Agent step 2 ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


No tool requests. Finishing.

=== Final Answer ===

Based on the repository content, here's a student-friendly summary:

## **1) What it is**
Requests is a simple, elegant Python library for making HTTP requests. It's one of the most popular Python packages with 30M+ downloads per week and is used by over 1 million repositories.

## **2) Typical use cases**
- Making GET, POST, PUT, DELETE requests to web APIs
- Downloading web content and streaming data
- Handling authentication (Basic, Digest)
- Working with JSON APIs
- Uploading files
- Managing sessions and cookies

## **3) How to install**
- `python -m pip install requests`
- Requires Python 3.9 or higher

## **4) Best practices / cautions**
- Use the built-in `.json()` method instead of manual parsing
- Take advantage of connection pooling for better performance
- Always set timeouts to avoid hanging requests
- Use sessions for cookie persistence across requests
- Library handles SSL/TLS verification automatically
- Supports inter